# Chapter 2 Practical 03: User Profile and Ranking

Learning objectives:
- Build a user profile from liked movies.
- Compare average, rating-weighted, normalized, implicit-feedback, and temporal-decay profiles.
- Recommend unseen movies.
- Explain recommendations using overlapping features.

Slide connection: user profiles, similarity matching, ranking, and Top-N recommendation.


Load movie metadata and a small set of user interactions.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()

interactions = pd.read_csv(DATA_DIR / "user_interactions_chapter2.csv")
interactions.head()


Create an item-feature matrix from genres, directors, and text. This gives the user profile a mix of structured and textual evidence.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

movies["feature_text"] = (
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
)
tfidf = TfidfVectorizer(stop_words="english")
text_features = tfidf.fit_transform(movies["feature_text"]).toarray()

numeric = MinMaxScaler().fit_transform(movies[["duration_min", "rating", "family_friendly"]])
feature_names = list(tfidf.get_feature_names_out()) + ["duration_scaled", "rating_scaled", "family_friendly"]
item_features = pd.DataFrame(
    np.hstack([text_features, numeric]),
    index=movies["title"],
    columns=feature_names,
)
item_features.iloc[:5, -8:].round(2)


For a simple average profile, each liked item has the same influence.


In [ ]:
user_id = "U1"
user_history = interactions[interactions["user_id"] == user_id].copy()
liked_titles = user_history["title"].tolist()

average_profile = item_features.loc[liked_titles].mean(axis=0)
average_profile.sort_values(ascending=False).head(10).round(3)


A rating-weighted profile gives stronger liked items more influence.


In [ ]:
weights_rating = user_history.set_index("title")["rating"]
rating_weighted_profile = item_features.loc[liked_titles].mul(weights_rating, axis=0).sum() / weights_rating.sum()
rating_weighted_profile.sort_values(ascending=False).head(10).round(3)


A rating-normalized profile centers ratings around the user's average. This reduces the effect of users who rate everything high.


In [ ]:
normalized_weights = weights_rating - weights_rating.mean()
if normalized_weights.abs().sum() == 0:
    normalized_weights = weights_rating / weights_rating.sum()

rating_normalized_profile = item_features.loc[liked_titles].mul(normalized_weights, axis=0).sum()
rating_normalized_profile.sort_values(ascending=False).head(10).round(3)


Implicit feedback can combine clicks, likes, and watch time. Temporal decay gives recent interactions more weight.


In [ ]:
max_duration = movies.set_index("title")["duration_min"]
history = user_history.set_index("title")
watch_ratio = history["watch_minutes"] / max_duration.loc[history.index]
implicit_weight = 0.2 * history["clicked"] + 0.5 * history["liked"] + 0.3 * watch_ratio
temporal_decay = np.exp(-history["days_ago"] / 30)
final_weight = implicit_weight * temporal_decay

profile_temporal = item_features.loc[history.index].mul(final_weight, axis=0).sum() / final_weight.sum()
pd.DataFrame({
    "implicit_weight": implicit_weight.round(3),
    "temporal_decay": temporal_decay.round(3),
    "final_weight": final_weight.round(3),
})


Recommend unseen movies by comparing each item vector to the user profile.


In [ ]:
def recommend_from_profile(profile, seen_titles, top_n=5):
    candidate_features = item_features.drop(index=seen_titles)
    scores = cosine_similarity(candidate_features, profile.values.reshape(1, -1)).ravel()
    results = pd.DataFrame({"title": candidate_features.index, "score": scores})
    return results.sort_values("score", ascending=False).head(top_n)

recommendations = recommend_from_profile(profile_temporal, liked_titles)
recommendations.round(3)


We can explain each recommendation by showing the strongest features shared by the user profile and the movie.


In [ ]:
def explain_recommendation(title, profile, top_n=6):
    contribution = item_features.loc[title] * profile
    return contribution.sort_values(ascending=False).head(top_n).round(3)

best_title = recommendations.iloc[0]["title"]
print(f"Explanation for {best_title}:")
explain_recommendation(best_title, profile_temporal)


## What did we learn?

- A user profile is a vector summarizing what the user liked.
- Different weighting choices create different profiles.
- Recommendations become more transparent when we inspect shared high-weight features.

Exercises:
1. Change `user_id` to `U2` or `U3` and compare the profile terms.
2. Increase or decrease the temporal decay speed. Which recommendations change?
